In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster")
system("conda install conda-forge::r-fnn")
system("conda install conda-forge::r-factominer conda-forge::r-caret conda-forge::r-factoextra conda-forge::r-rlang")
system("conda install conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools")



In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)

library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

library(cluster)
library(data.table)
library(bioregion)

library(factoextra)

library(FNN)

In [ ]:
global_topo_tiff_gz <- "global_topo.tiff.gz"

# nchar(global_topo_tiff)
filename.length <- nchar(global_topo_tiff_gz)

# Get the name without the extension ("." + 2 letters)
start <- 1
end <- filename.length-3
global_topo_tiff <- substr(global_topo_tiff_gz,start, end)


In [ ]:
# CREATE DIRECTORIES
data.directory <- "data"
ifelse(!dir.exists(file.path(data.directory)),
        dir.create(file.path(data.directory)),
        "Directory Exists")

output.directory <- "outputs"
ifelse(!dir.exists(file.path(output.directory)),
        dir.create(file.path(output.directory)),
        "Directory Exists")

# MOVE TO DIRECTORIES 

bathy_file <- "global_topo_1min_topo_19_1.nc"

ifelse(test = file.exists(paste(data.directory, global_topo_tiff_gz, sep = "/")),
       yes = "File Exists",
       no = file.rename(from=global_topo_tiff_gz,
            to=paste(data.directory, global_topo_tiff_gz, sep = "/"))
       )

ifelse(test = file.exists(paste(data.directory, global_topo_tiff, sep = "/")),
       yes = "File Exists",
       no = file.rename(from=global_topo_tiff,
            to=paste(data.directory, global_topo_tiff, sep = "/"))
       )

ifelse(test = file.exists(paste(data.directory, bathy_file, sep = "/")),
       yes = "File Exists",
       no = file.rename(from=bathy_file,
            to=paste(data.directory, bathy_file, sep = "/"))
       )
bathy_file <- paste(data.directory, bathy_file, sep = "/")


## NetCDF (Copernicus Data Files)

In [ ]:
# Open the NetCDF file
nc_file <- nc_open(bathy_file)
print(nc_file)

In [ ]:
# Get coordinates variables
longitude <- ncvar_get(nc_file,"lon")
latitude <- ncvar_get(nc_file,"lat")
z <- ncvar_get(nc_file,"z")

fillvalue <- ncatt_get(nc_file, "z", "_FillValue") # The fill value (aka, the no data value) is -9999.

z[z == fillvalue$value] <- NA
nc.slice.min80 <- z[,1]


### Plots

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================

Bathy <- rast(bathy_file)

png <- 1
if (png) {
  png(filename = "outputs/Fig1-Bathy.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()


### Convert NetCDF to CSV
Based on [Copernicus Marine Services](https://help.marine.copernicus.eu/en/articles/6328012-how-to-convert-netcdf-to-csv-using-r)

In [ ]:
#  Extract the coordinates

dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)

coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))

names(depth_matrix) <- c("lon", "lat", "depth")


In [ ]:
nc_close(nc_file)

# csv_fname <- paste(output.directory,"netcdf_depth.csv", sep="/")
# write.table(depth_matrix, csv_fname, row.names=FALSE, sep=";")
# print("The matrix was exported in CSV format.")

# NASA Data - SST

In [ ]:
# 2.2 ANALYSIS ----
SST_file <- "data/2008/AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.nc"
nc_sst_file <- NULL
if(file.exists(SST_file)){
   nc_sst_file <- nc_open(SST_file)
   print(nc_sst_file)
} else {
       print(paste(nc_sst_file,"does not exist"))
}

Based on the time coverage shown above:
```
...
    61 global attributes:
        product_name: AQUA_MODIS.20071101_20071130.L3m.MO.SST.sst.9km.nc
        instrument: MODIS
        ...
        time_coverage_start: 2007-11-01T00:10:01.000Z
        time_coverage_end: 2007-12-01T02:05:00.000Z
        ...
...
```
we can see that the time coverage of the satellite extends until the first hours of the next month. Hence, the reason why the API included november monthly measures. 

In [ ]:
#  Extract the coordinates
dim_sst_lon <- ncvar_get(nc_sst_file, "lon")
dim_sst_lat <- ncvar_get(nc_sst_file, "lat")
dim_sst <- ncvar_get(nc_sst_file, "sst", raw_datavals = TRUE)

# Get attributes
fill_value <- ncatt_get(nc_sst_file, "sst", "_FillValue")$value ; fill_value
scale      <- ncatt_get(nc_sst_file, "sst", "scale_factor")$value ; scale

sst_raw <- ncvar_get(nc_sst_file, "sst", raw_datavals = TRUE)
sst_raw[sst_raw == fill_value] <- NA
sst <- sst_raw * scale

sst_coords <- expand.grid(dim_sst_lon, dim_sst_lat)
sst_matrix <- cbind(sst_coords, as.vector(dim_sst))
names(sst_matrix) <- c("lon", "lat", "sst")


In [ ]:
# Close file
nc_close(nc_sst_file)


# Ice concentration

In [ ]:
# 2.2 DOWNLOAD COPERNICUS NetCDF ----

# Sea ice area & Sea ice concentration estimates as retrieved by the algorithm,
# and that were edited away by the various filters
# system("python API_Copernicus.py")

# OR

# Use 'Copernicus marine data' output as an input
# ice_netcdf <- "./data/osisaff_obs-si_glo_phy_sic-south_my_amsr_cdr_P1D-m_1778859355280.nc"
ice_netcdf <- 'data/Copernicus marine data.netcdf'
nc_iceC_file <- nc_open(ice_netcdf)


In [ ]:
#  Extract the coordinates
dim_ice_lon <- ncvar_get(nc_iceC_file, "longitude") # nc_iceC_file$dim[[3]]$vals . longitude
dim_ice_lat <- ncvar_get(nc_iceC_file, "latitude") # nc_iceC_file$dim[[2]]$vals . latitude
dim_ice_time <- ncvar_get(nc_iceC_file, "time") # nc_iceC_file$dim[[1]]$vals . time
dim_ice <- ncvar_get(nc_iceC_file, "ice_conc")
dim_raw_ice <- ncvar_get(nc_iceC_file, "raw_ice_conc_values")

#sea_water_potential_concentration
T <- c("ice_conc", "raw_ice_conc_values")
t_units <- ncatt_get(nc_iceC_file, "time", "units")
# convert time -- split the time units string into fields
t_ustr <- strsplit(t_units$value, " ")
t_dstr <- strsplit(unlist(t_ustr)[3], "-")
date <- ymd(t_dstr) + dseconds(dim_ice_time)

T_array_ice <- dim_ice
# Quick map plot

# set the time step
t <- 1 #temperature on 2003-01-01
T_slice <- T_array_ice[,,t]

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/IceConcentration.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/IceConcentration.pdf")
}

# Plot Ice Concentration
image(x = dim_ice_lon, y = dim_ice_lat, z = T_slice, col = rev(brewer.pal(10,"RdBu")))

# Save the plot ---
dev.off()

In [ ]:
grid <- expand.grid(lon=dim_ice_lon, lat=dim_ice_lat)  #create a set of lonxlat pairs of values, one for each element in the Temp_array

cutpts <- c(12,13,14,15,16,17,18,19,20)   # set colorbar
levelplot(T_slice ~ lon * lat,
          data=grid, region=TRUE,
          pretty=T, at=cutpts, cuts=9,
          col.regions=(rev(brewer.pal(9,"RdBu"))), contour=0,
          xlab = "Longitude", ylab = "Latitude",
          main = "Sea Water Potential Temperature (°C)"
          )

In [ ]:
ice_raw <- ncvar_get(nc_iceC_file, "ice_conc", raw_datavals = TRUE)

# Get attributes
fill_value <- ncatt_get(nc_iceC_file, "ice_conc", "_FillValue")$value ; fill_value
scale      <- ncatt_get(nc_iceC_file, "ice_conc", "scale_factor")$value ; scale

ice_raw[ice_raw == fill_value] <- NA
ice <- ice_raw * scale

ice_coords <- expand.grid(dim_ice_lon, dim_ice_lat,date)
ice_matrix <- cbind(ice_coords, as.vector(dim_ice))
names(ice_matrix) <- c("lon", "lat", "time", "iceC")


In [ ]:
nc_close(nc_iceC_file)

In [ ]:
# If you already have every CSV you need
depth_matrix <- read.table("outputs/netcdf_depth.csv", header = FALSE)
sst_matrix <- read.table("outputs/AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.csv", header = FALSE)
ice_matrix <- read.table("outputs/Ice_CopernicusMarineData.csv", header = FALSE)

# Function to replace NaN with column median

df_clean <- lapply(ice_matrix, function(iceC) {
    if (is.numeric(iceC)) {
      # Compute median excluding NA and NaN
      med <- median(iceC, na.rm = TRUE)
      
      # Replace NaN values with median
      iceC[is.nan(iceC)] <- med
    }
    return(iceC)}
)


# Concatenate the Dataframes

In [ ]:
# ----------------
# Filter Out the outerbound values

## 1. Depth
Antarctic_depth_coords <- depth_matrix %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60
  )

## 2. SST
Antarctic_sst_coords <- sst_matrix %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60
  )

## 3. IceConc (Ice Concentration)
Antarctic_iceC_coords <- as.data.frame(df_clean) %>%
  dplyr::mutate(
    month = format(time, "%m") # Apply month format
  ) %>%
  dplyr::filter(
    lon >= 30, lon <= 150, lat >= -70, lat <= -60,
    (month %in% c("01", "02", "03", "12")) | (month == "04" & format(time, "%d") <= "01")
  ) %>%
    dplyr::group_by(lon, lat) %>% # Groups ONLY BY COORDINATES
    dplyr::summarise(
    ice_mean = mean(iceC, na.rm = TRUE), # Compute the average Ice concentration in summer
    .groups = "drop"
  )


In [ ]:
# ----------------
# Interpolation
## 1. Create a new grid
new.grid.lon <- seq(from=30.0, to=150.0, by=0.05)
new.grid.lat <- seq(from=-70.0, to=-60.0, by=0.05)
new.grid <- expand.grid(lon=new.grid.lon, lat=new.grid.lat)
head(new.grid)

## 2. Add Bathymetry (depth) + Sea Surface Temperature (sst) + IceConc (iceC)
### +  Fill the added columns  
new.grid$sst <- NA
new.grid$depth <- NA
new.grid$iceC <- NA


In [ ]:
grid   <- as.data.table(new.grid)
sst_dt <- as.data.table(Antarctic_sst_coords)
depth_dt <- as.data.table(Antarctic_depth_coords)
ice_dt <- as.data.table(Antarctic_iceC_coords)

In [ ]:
# ----------------
# Find nearest neighbor for each grid point
# + Assign interpolated values

## 1. DEPTH
nn <- get.knnx(
  data  = as.matrix(depth_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)
grid[, depth := depth_dt$depth[nn$nn.index]]

## 2. SST
# Find nearest neighbor for each grid point
nn <- get.knnx(
  data  = as.matrix(sst_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)
grid[, sst := sst_dt$sst[nn$nn.index]]

## 3. ICE CONCENTRATION
nn <- get.knnx(  data  = as.matrix(
    ice_dt[, .(lon, lat)]),
    query = as.matrix(grid[, .(lon, lat)]),
    k = 1
)
grid[, iceC := ice_dt$ice_mean[nn$nn.index]]
grid %>% rename("ice_mean" = "iceC")


# Statistical Analyses

In [ ]:
install.packages("remotes")
remotes::install_github("bioRgeo/bioregion")

# install.packages("devtools")
# devtools::install_github("bioRgeo/bioregion")

In [ ]:
# If you already have every CSV you need
grid <- data.table::fread(file = "outputs/PelagicData.csv", header = TRUE, sep=";")

# Default value of dissimilarity matrix
# clean SST flag
grid[sst == -32767, sst := NA]

rdm.rows <- sample(nrow(grid), 900, replace = FALSE)
X <- grid[rdm.rows, .(sst, depth, iceC)]

# compute Gower distances (small subset!)
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)

In [ ]:
# Custom function of dissimilarity matrix
dim(distance.matrix)

In [ ]:
clara.object <- bioregion::nhclu_clara(
    dissimilarity = distance.matrix, n_clust = seq(from = 2, to = dim(distance.matrix)[1] , by= 1))
                                       


In [ ]:
names(clara.object)

In [ ]:
# List the attributes and sub-attributes

i <- 1
for(listVar in names(clara.object)){
    cat(i, listVar,"\n")
    # print(clara.object[[listVar]])
    for(listNames in names(clara.object[[listVar]])){
        cat("Attr:", listNames, ":", names(clara.object[[listVar]][[listNames]]),"\n")
        # head(clara.object[[listVar]][[listNames]])
    }
    i <- i + 1
}

In [ ]:
head(clara.object$clusters)

In [ ]:
# 1. Clean data
df_clean <- grid %>% mutate(sst = ifelse(sst == -32767, NA, sst))

# 2. Choose k
k_values <- 2:10
sil_width <- sapply(k_values, function(k) {
    nhclu_clara(df_clean, n_clust )$silinfo$avg.width
})

best_k <- k_values[which.max(sil_width)]

# 3. Final model
final_model <- clara(df_clean, k = best_k)
df_clean$cluster <- final_model$clustering

In [ ]:
sil_width
k_values
best_k

In [ ]:
# Compute Gower distance once
gower_dist <- daisy(df_clean, metric = "gower")

# Try different k values
k_values <- 2:10
sil_width <- numeric(length(k_values))

for (i in seq_along(k_values)) {
  k <- k_values[i]
  clara_fit <- clara(df_clean, k = k)
  sil_width[i] <- clara_fit$silinfo$avg.width
}

# Plot
plot(k_values, sil_width, type = "b",
     xlab = "Number of clusters (k)",
     ylab = "Average silhouette width")

# Best k
best_k <- k_values[which.max(sil_width)]

In [ ]:
fviz_nbclust(df_clean, clara, method = "gap_stat", k.max = 10)

In [ ]:
gower_dist <- daisy(df_clean,
                    metric = "gower",
                    weights = c(lon = 1, lat = 1, sst = 2, depth = 2, iceC = 2))